In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import zscore


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'
OUTPUT_DIR = 'results'

In [4]:
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok = True)

In [5]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [6]:
metric_cols = [
    "Accuracy",
    "Weighted Accuracy",
    "Time",
    "Monotonicity",
    "Separability",
    "Linearity"
]

In [7]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    # if 'dementia' not in file:

        # Process texts.
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w.lower() not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=[ 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Total Documents,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,4978,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,13069,0.498065,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,23700,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,3603,0.398338,4.360843,0.035360,0.267681,46.278934,17.0
4,dementiaAudio,549,0.453092,3.480664,0.026194,0.386441,116.338798,105.0
5,huffPostNews,189815,0.390427,4.199670,0.023536,0.238591,25.163032,23.0
6,medicalAbstracts,14438,0.308262,5.109807,0.021201,0.263213,205.660064,200.0
7,simSUM,10000,0.124588,4.106722,0.012611,0.381137,104.821400,103.0
8,syntheticCareHomeNurseNotes,5783,0.340652,4.937623,0.027334,0.277734,26.532596,24.0
9,yahoo,87362,0.424969,3.922505,0.029172,0.255604,47.845493,42.0


In [8]:
model_dict = {
    'cross-encoder/nli-deberta-v3-small': 'model1',
    'typeform/distilbert-base-uncased-mnli': 'model2',
    'valhalla/distilbart-mnli-12-3': 'model3'
}

def make_stripped_csv(all_mean_df):
    new_rows = []
    for index, row in all_mean_df.iterrows():
        temp_name = row['metric']
        split_name = temp_name.split('__')
        # single model comparisons
        if split_name[1] == 'prompt1_prompt2':
            if len(split_name[0].split('_')) == 1:

                row['metric'] = model_dict[split_name[0]]
                new_rows.append(row)
        # single prompt comparisons
        if split_name[0] == 'cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3':
            if len(split_name[1].split('_')) == 1:
                row['metric'] = split_name[1]
                new_rows.append(row)

    final_all_mean_df = pd.DataFrame(new_rows)
    return final_all_mean_df

In [9]:
def make_non_normalized_dfs(input_folder, output_file_name):
    all_temp_dfs = []
    for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}'):
        if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}'):
            combination_splits = combination.split('_')
            dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}/{combination}_ksc_metrics_measures.csv')
            temp_df = make_stripped_csv(temp_df)
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Dataset 1'] = dataset1
            grouped_df['Dataset 2'] = dataset2
            grouped_df['Repetitions'] = repetitions
            grouped_df['Metric'] = grouped_df.index
            grouped_df.reset_index(inplace=True)
            grouped_df.drop(columns='metric', inplace=True)
            all_temp_dfs.append(grouped_df)

    all_dfs = pd.concat(all_temp_dfs)

    print(# All datasets should have been compared the same number of times for this section to work.
    Counter(list(all_dfs['Dataset 1']) + list(all_dfs['Dataset 2'])))

    temp_dataset_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = all_dfs[
                (all_dfs['Dataset 1'] == dataset) | 
                (all_dfs['Dataset 2'] == dataset)
            ].copy()
        temp_df = temp_df.groupby('Metric')[metric_cols].mean()
        temp_df['Dataset'] = dataset
        temp_dataset_dfs.append(temp_df)
    dataset_df = pd.concat(temp_dataset_dfs)
    dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}.xlsx')

    mean_dataset_df = dataset_df.groupby('Metric')[metric_cols].mean()
    mean_dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}Mean.xlsx')

    return dataset_df, mean_dataset_df

In [10]:
ksc_dataset_df, ksc_mean_dataset_df = make_non_normalized_dfs('ksc', 'kscDataset')
ksc_synth_dataset_df, ksc_synth_mean_dataset_df = make_non_normalized_dfs('ksc_synth', 'kscSynthDataset')

Counter({'atis': 128, 'banking77': 128, 'clinc150': 128, 'clinicalDialogueSummarizations': 128, 'huffPostNews': 128, 'medicalAbstracts': 128, 'simSUM': 128, 'syntheticCareHomeNurseNotes': 128, 'yahoo': 128})
Counter({'atis': 32, 'banking77': 32, 'clinc150': 32, 'clinicalDialogueSummarizations': 32, 'huffPostNews': 32, 'medicalAbstracts': 32, 'simSUM': 32, 'syntheticCareHomeNurseNotes': 32, 'yahoo': 32})


In [11]:
ksc_dataset_df['Type'] = 'KSC'
ksc_synth_dataset_df['Type'] = 'KSC_Synth'

In [12]:
all_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Dataset', 'Metric'])[metric_cols].mean()
all_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Metric'])[metric_cols].mean()
all_type_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Metric'])[metric_cols].mean()
all_type_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Dataset', 'Metric'])[metric_cols].mean()

In [13]:
all_dataset_df.to_excel(f'./{OUTPUT_DIR}/allDataset.xlsx')
all_mean_df.to_excel(f'./{OUTPUT_DIR}/allDatasetMean.xlsx')

In [14]:
all_type_mean_df

Accuracy  Weighted Accuracy      Time  Monotonicity  \
Type      Metric                                                         
KSC       model1   0.747129           0.705515  0.037804      0.605309   
          model2   0.697910           0.659686  0.048713      0.491423   
          model3   0.744324           0.700437  0.017578      0.597903   
          prompt1  0.742503           0.699288  0.019389      0.601114   
          prompt2  0.744461           0.704085  0.019165      0.600264   
          prompt3  0.727482           0.685800  0.020052      0.570590   
          prompt4  0.723627           0.682394  0.019250      0.552382   
          prompt5  0.720988           0.678489  0.019800      0.569757   
KSC_Synth model1   0.645257           0.614483  0.045969      0.349627   
          model2   0.643693           0.609316  0.054993      0.395420   
          model3   0.677996           0.650588  0.020802      0.417361   
          prompt1  0.653283           0.624122  0.022930      0.398294   
          prompt2  0.653849           0.619040  0.022623      0.415316   
          prompt3  0.606193           0.580466  0.023823      0.304776   
          prompt4  0.602983           0.572667  0.022771      0.311309   
          prompt5  0.657357           0.626176  0.023569      0.403836   

                   Separability  Linearity  
Type      Metric                            
KSC       model1       0.437700   0.642178  
          model2       0.300252   0.524601  
          model3       0.397908   0.631096  
          prompt1      0.410472   0.632137  
          prompt2      0.419348   0.631041  
          prompt3      0.384148   0.604675  
          prompt4      0.364230   0.586249  
          prompt5      0.381299   0.603949  
KSC_Synth model1       0.147988   0.371348  
          model2       0.195749   0.425198  
          model3       0.241794   0.431749  
          prompt1      0.205120   0.425287  
          prompt2      0.244703   0.449184  
          prompt3      0.146005   0.338782  
          prompt4      0.110171   0.326358  
          prompt5      0.207168   0.437449

In [15]:
all_type_dataset_df

Accuracy  Weighted Accuracy      Time  \
Type      Dataset Metric                                           
KSC       atis    model1   0.783903           0.739277  0.050168   
                  model2   0.755683           0.710723  0.063765   
                  model3   0.744362           0.697865  0.023695   
                  prompt1  0.768669           0.723414  0.025954   
                  prompt2  0.757224           0.714537  0.025533   
...                             ...                ...       ...   
KSC_Synth yahoo   prompt1  0.639298           0.625916  0.021336   
                  prompt2  0.624945           0.585443  0.021038   
                  prompt3  0.582188           0.567616  0.022029   
                  prompt4  0.568878           0.543791  0.021153   
                  prompt5  0.618379           0.592620  0.021827   

                           Monotonicity  Separability  Linearity  
Type      Dataset Metric                                          
KSC       atis    model1       0.683081      0.534482   0.721132  
                  model2       0.616158      0.416829   0.657725  
                  model3       0.606566      0.390969   0.633793  
                  prompt1      0.657954      0.479795   0.692213  
                  prompt2      0.633745      0.453011   0.668056  
...                                 ...           ...        ...  
KSC_Synth yahoo   prompt1      0.323413      0.144351   0.332707  
                  prompt2      0.417073      0.246459   0.475165  
                  prompt3      0.271287      0.101405   0.293440  
                  prompt4      0.305567      0.153538   0.313092  
                  prompt5      0.308126      0.111278   0.354999  

[144 rows x 6 columns]

In [16]:
all_dataset_df

Accuracy  Weighted Accuracy      Time  Monotonicity  \
Dataset Metric                                                         
atis    model1   0.759993           0.714374  0.057350      0.611402   
        model2   0.693461           0.656541  0.070240      0.483748   
        model3   0.652247           0.620286  0.025773      0.387281   
        prompt1  0.712873           0.672029  0.028766      0.542234   
        prompt2  0.687751           0.652842  0.028131      0.458559   
...                   ...                ...       ...           ...   
yahoo   prompt1  0.665429           0.639389  0.019938      0.413322   
        prompt2  0.665146           0.626160  0.019734      0.467089   
        prompt3  0.635151           0.610508  0.020565      0.371631   
        prompt4  0.624445           0.591333  0.019817      0.387017   
        prompt5  0.664886           0.632060  0.020360      0.425334   

                 Separability  Linearity  
Dataset Metric                            
atis    model1       0.397782   0.645101  
        model2       0.258053   0.509759  
        model3       0.204363   0.399085  
        prompt1      0.357999   0.578498  
        prompt2      0.308176   0.486821  
...                       ...        ...  
yahoo   prompt1      0.215432   0.432225  
        prompt2      0.282251   0.512757  
        prompt3      0.183810   0.398425  
        prompt4      0.217835   0.408194  
        prompt5      0.225308   0.463423  

[72 rows x 6 columns]

In [18]:
all_mean_df = all_mean_df.round(4)

In [19]:
all_mean_df

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity
Metric,,,,,,
model1,0.6962,0.6600,0.0419,0.4775,0.2928,0.5068
model2,0.6708,0.6345,0.0519,0.4434,0.2480,0.4749
model3,0.7112,0.6755,0.0192,0.5076,0.3199,0.5314
prompt1,0.6979,0.6617,0.0212,0.4997,0.3078,0.5287
prompt2,0.6992,0.6616,0.0209,0.5078,0.3320,0.5401
prompt3,0.6668,0.6331,0.0219,0.4377,0.2651,0.4717
prompt4,0.6633,0.6275,0.0210,0.4318,0.2372,0.4563
prompt5,0.6892,0.6523,0.0217,0.4868,0.2942,0.5207
